In [1]:
# Install required packages
!pip install azure-identity openai matplotlib --quiet


In [9]:
from azure.identity import InteractiveBrowserCredential
from openai import AzureOpenAI
import base64

# Step 1: Azure AD login via browser popup
credential = InteractiveBrowserCredential()
token = credential.get_token("https://cognitiveservices.azure.com/.default").token


In [10]:
# Step 2: Create AzureOpenAI client with GPT-4o deployment
client = AzureOpenAI(
    api_version="2024-12-01-preview",
    azure_endpoint="https://yashcompliancegovernance.openai.azure.com/",  # 👈 your endpoint
    azure_deployment="gpt-4o",  # 👈 your deployed model name in Azure
    azure_ad_token=token  # 👈 from InteractiveBrowserCredential
)


In [11]:
# === Encode an image (e.g., SHAP or LIME plot) ===
image_path = r".\images\breast_cancer\waterfall\sample_0.png"  # or any other plot image you’ve saved
with open(image_path, "rb") as f:
    image_base64 = base64.b64encode(f.read()).decode("utf-8")


In [12]:
# Step 4: Send image to GPT-4o with a prompt
response = client.chat.completions.create(
    model="gpt-4o",  # Required by SDK; Azure uses deployment name
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Explain this SHAP plot like I’m a domain expert."},
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_base64}"}}
            ]
        }
    ]
)

# Step 5: Print the response
print("🧠 GPT-4o response:\n", response.choices[0].message.content)


🧠 GPT-4o response:
 This SHAP (SHapley Additive exPlanations) plot summarizes the contribution of features to a predictive model's output for a single data point. For a domain expert, here’s what the plot is saying:

---

### Key Details:
1. **Output prediction (`f(x)`):** 
   - The model predicted a value of **0.968** for this data point (shown at the top of the plot).
   - The base value (**E[f(x)]**) is **0.632**, which is the average model prediction if we ignore feature-specific information. This means the model started at a baseline value of 0.632, and then made adjustments based on feature values.

2. **Feature contributions:**
   - Each red bar represents an individual feature’s positive contribution to pushing the prediction closer to **0.968**. Features are ranked based on their impact in descending order.
   - Gray features represent those whose contributions are negligible (e.g., **21 other features** in the plot).

---

### Feature Analysis:
- **Top contributors:**
  - **W

In [19]:
import os
import json
import base64
import time
from tqdm import tqdm
from azure.identity import InteractiveBrowserCredential
from openai import AzureOpenAI

# ---------------------------------------------------------
# 1. Configuration
# ---------------------------------------------------------
AZURE_ENDPOINT = "https://yashcompliancegovernance.openai.azure.com/"
DEPLOYMENT_NAME = "gpt-4o"
API_VERSION = "2024-12-01-preview"

# Updated Paths: Assumes these are in the SAME directory as this notebook
METADATA_PATH = "dataset_metadata.json"
OUTPUT_DATASET_PATH = "finetuning_dataset.jsonl"

# --- THE FIX: Map JSON labels to Real Folder names ---
# The generator saved specific types (e.g., "lime_vertical") in the JSON,
# but stored the images in general folders (e.g., "lime"). This maps them back.
FOLDER_MAPPING = {
    # Tabular Maps
    "shap_waterfall": "waterfall",
    "shap_bar": "bar",
    "shap_force": "force",
    "shap_decision": "decision",
    "lime_standard": "lime",
    "lime_vertical": "lime",
    "lime_lollipop": "lime",
    "lime_tabular": "lime",
    
    # Text Maps
    "lime_text": "lime",
    "shap_text_bar": "shap_bar",
    
    # Globals (Usually match, but added for safety)
    "global_beeswarm": "global_beeswarm",
    "global_dependence": "global_dependence"
}

print("Please sign in via the browser popup...")
credential = InteractiveBrowserCredential()
token = credential.get_token("https://cognitiveservices.azure.com/.default").token

client = AzureOpenAI(
    api_version=API_VERSION,
    azure_endpoint=AZURE_ENDPOINT,
    azure_deployment=DEPLOYMENT_NAME,
    azure_ad_token=token
)
print("✅ Authenticated successfully!")

# ---------------------------------------------------------
# 2. Path Logic
# ---------------------------------------------------------
def get_local_image_path(entry):
    """
    Constructs the correct local path ./images/... handling folder mismatches.
    """
    # 1. Get raw path from JSON (e.g. "breast_cancer/shap_waterfall/sample_0.png")
    raw_path = entry['image_rel_path'].replace("\\", "/") 
    
    # 2. Split into parts: [domain, plot_type, filename]
    # e.g. ["breast_cancer", "shap_waterfall", "sample_0.png"]
    parts = raw_path.split("/")
    
    # Clean out any "dataset" or "images" prefix if present in the JSON path
    parts = [p for p in parts if p not in ["dataset", "images"]]
    
    # 3. APPLY MAPPING FIX
    # If the middle part (folder) is in our mapping, swap it.
    # e.g. "shap_waterfall" -> "waterfall"
    if len(parts) >= 2:
        folder_name = parts[-2] # The folder containing the image
        if folder_name in FOLDER_MAPPING:
            parts[-2] = FOLDER_MAPPING[folder_name]
            
    # 4. Construct Path with ./images prefix (Since images folder is next to notebook)
    # Result: .\images\breast_cancer\waterfall\sample_0.png
    local_path = os.path.join(".", "images", *parts)
    
    return os.path.normpath(local_path)

def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

def construct_system_prompt(domain):
    if "cancer" in domain or "medical" in domain:
        return "You are an expert Medical AI Consultant. Interpret this clinical risk model visualization accurately for a doctor."
    elif "finance" in domain:
        return "You are a Senior Financial Risk Analyst. Explain this credit scoring model decision to a loan officer."
    elif "text" in domain:
        return "You are an NLP Interpretability Expert. Explain which words drove the text classification."
    else:
        return "You are an expert in Explainable AI (XAI). Interpret this feature attribution plot."

def generate_teacher_explanation(entry):
    image_path = get_local_image_path(entry)
    
    # Debug print only if it fails
    if not os.path.exists(image_path):
        print(f"⚠️ Image NOT found at: {image_path}")
        return None

    try:
        base64_image = encode_image(image_path)
    except Exception as e:
        print(f"❌ Read Error: {e}")
        return None
    
    ground_truth_context = f"""
    CONTEXT DATA (Ground Truth):
    - Domain: {entry['domain']}
    - Model Prediction: {entry['model_output']}
    - Top Feature Attributions: {json.dumps(entry['top_attributions'])}
    
    TASK:
    Generate a concise, natural language explanation of this prediction based on the provided plot and the Context Data above.
    1. Start by stating the model's prediction.
    2. Identify the most critical features driving this specific prediction using the Context Data values.
    3. Explain the direction of influence (e.g., "High Age increased risk").
    4. Do not mention "Context Data" or "JSON" in the output—write as if you are analyzing the case directly.
    """

    try:
        response = client.chat.completions.create(
            model=DEPLOYMENT_NAME,
            messages=[
                {
                    "role": "system", 
                    "content": construct_system_prompt(entry['domain'])
                },
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": ground_truth_context},
                        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{base64_image}"}}
                    ]
                }
            ],
            temperature=0.2, 
            max_tokens=300
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"\n❌ API Error processing {entry['id']}: {e}")
        return None

# ---------------------------------------------------------
# 3. Main Loop
# ---------------------------------------------------------
def main():
    if not os.path.exists(METADATA_PATH):
        print(f"Error: Metadata file not found at {METADATA_PATH}")
        return

    with open(METADATA_PATH, "r") as f:
        metadata_registry = json.load(f)
    
    print(f"📂 Loaded {len(metadata_registry)} entries.")
    
    processed_ids = set()
    if os.path.exists(OUTPUT_DATASET_PATH):
        with open(OUTPUT_DATASET_PATH, "r") as f:
            for line in f:
                try: processed_ids.add(json.loads(line)['id'])
                except: pass
    
    print(f"🔄 Resuming... {len(processed_ids)} already processed.")

    with open(OUTPUT_DATASET_PATH, "a") as out_f:
        for entry in tqdm(metadata_registry, desc="Processing"):
            if entry['id'] in processed_ids:
                continue 
            
            explanation = generate_teacher_explanation(entry)
            
            if explanation:
                # Save path for fine-tuning. 
                # Standard format usually expects "images/folder/file.png"
                clean_path = get_local_image_path(entry).replace(".\\", "").replace("\\", "/")
                
                finetuning_entry = {
                    "id": entry['id'],
                    "image": clean_path,
                    "conversations": [
                        {"from": "human", "value": "<image>\nExplain the model's prediction for this case."},
                        {"from": "gpt", "value": explanation}
                    ]
                }
                out_f.write(json.dumps(finetuning_entry) + "\n")
                out_f.flush()
            
            time.sleep(0.5)

    print(f"\n🎉 SUCCESS! Dataset generated at: {OUTPUT_DATASET_PATH}")

if __name__ == "__main__":
    main()

Please sign in via the browser popup...
✅ Authenticated successfully!
📂 Loaded 700 entries.
🔄 Resuming... 0 already processed.


Processing: 100%|████████████████████████████████████████████████████████████████████| 700/700 [53:25<00:00,  4.58s/it]


🎉 SUCCESS! Dataset generated at: finetuning_dataset.jsonl
